# 11 国港口与管线再核查
只读当前发布库；完整性声明与证据局限见 COVERAGE_RECHECK.md。

In [1]:
from pathlib import Path
import json, duckdb
root = Path.cwd()
if not (root/'runtime/monitoring.duckdb').exists(): root = root.parent
audit = json.loads((root/'audits/port_and_pipeline_recheck_2026-10-16.json').read_text())
assert audit['complete_and_accurate_without_exceptions'] is False
audit['source_coverage']

{'reviewed_at_utc': '2026-10-16T20:07:17.129000+00:00', 'portwatch': {'url': 'https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/PortWatch_ports_database/FeatureServer/0/query?f=json&where=country+IN+%28%27Bahrain%27%2C%27Djibouti%27%2C%27Egypt%27%2C%27Iran%27%2C%27Iraq%27%2C%27Kuwait%27%2C%27Oman%27%2C%27Qatar%27%2C%27Saudi+Arabia%27%2C%27United+Arab+Emirates%27%2C%27Yemen%27%29&outFields=portid%2Cportname%2Ccountry%2Clat%2Clon&returnGeometry=false&resultRecordCount=1000&orderByFields=portid+ASC', 'sha256': '9cc52def0c15bf7b1ebea746390fa2496d68841bfe8bbf537a7ceadb714ad2f5', 'rows': 91, 'truncated': False, 'missing_source_ids': []}, 'wpi': {'scope_rows': 125, 'missing_reference_ids': [], 'source_caveat': 'US DOE NETL mirror fixed version, major ports only'}, 'facts': {'catalog_ids': 170, 'independent_statistics_nodes': 91, 'catalogue_only_nodes': 79, 'daily_records': 258349, 'first_day': '2019-01-01', 'last_day': '2026-10-09', 'nodes_with_daily_records': 91}, 'pipelines

In [2]:
with duckdb.connect(str(root/'runtime/monitoring.duckdb'), read_only=True) as con:
    ports = [json.loads(row[0]) for row in con.execute("SELECT payload FROM catalogs WHERE kind='ports'").fetchall()]
ports = [p for p in ports if p['country'] != 'Eritrea']
assert len(ports) == len({p['portid'] for p in ports}) == 170
required = ['name','name_cn','name_en','display_name','country','region','lat','lon','source_url']
{k: sum(p.get(k) in (None,'') for p in ports) for k in required}

{'name': 0, 'name_cn': 0, 'name_en': 0, 'display_name': 0, 'country': 0, 'region': 0, 'lat': 0, 'lon': 0, 'source_url': 0}

In [3]:
production = audit['unlocode']['production']
assert len(production['rows']) == 255
matched = sum(bool(row['matching_catalog_ids']) for row in production['rows'])
{'matched_reference_locations': matched, 'unresolved_reference_locations': 255-matched, 'confirmed_missing_facility_entries': len(audit['known_missing_facility_entries'])}

{'matched_reference_locations': 103, 'unresolved_reference_locations': 152, 'confirmed_missing_facility_entries': 9}

In [4]:
from collections import Counter
{'current_country_counts':dict(Counter(p['country'] for p in ports)), 'known_missing':audit['known_missing_facility_entries'], 'nearby_pairs_for_review':audit['profile']['nearby_pairs']}

{'current_country_counts': {'Djibouti': 4, 'Yemen': 11, 'Egypt': 36, 'Bahrain': 5, 'Iran': 23, 'Iraq': 8, 'Kuwait': 6, 'Qatar': 8, 'Saudi Arabia': 25, 'United Arab Emirates': 34, 'Oman': 10}, 'known_missing': [{'country': 'United Arab Emirates', 'name': '迪巴港（Dibba Port）', 'status': 'source-backed facility or project absent from current catalogue; no independent traffic invented', 'source_urls': ['https://fujairahport.ae/port-overview/dibba-port/', 'https://www.rakports.ae/ports/saqr-port-and-free-zone/']}, {'country': 'Saudi Arabia', 'name': '法拉桑客运港（Farasan ferry terminal）', 'status': 'source-backed facility or project absent from current catalogue; no independent traffic invented', 'source_urls': ['https://www.spa.gov.sa/en/N2163485', 'https://www.iaphworldports.org/memberports/saudi-ports-authority/']}, {'country': 'Iran', 'name': '伦格港（Bandar Lengeh）', 'status': 'source-backed facility or project absent from current catalogue; no independent traffic invented', 'source_urls': ['https: